# Iris Flower Classifier

A beginner-friendly walkthrough of a complete supervised-learning workflow with scikit-learn:

1. Load and explore the data
2. Split into train / test sets
3. Train two classifiers (Logistic Regression and Random Forest)
4. Evaluate with accuracy, a classification report, and a confusion matrix
5. Cross-validate and make a prediction on a new flower

The **Iris dataset** (150 flowers, 3 species, 4 measurements each) ships with scikit-learn, so no download is needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Load and explore the data

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Summary statistics and class balance
display(df.describe().round(2))
print("\nClass counts:")
print(df["species"].value_counts())

In [ ]:
# Pairwise relationships, coloured by species
sns.pairplot(df.drop(columns="target"), hue="species", corner=True)
plt.show()

Setosa is clearly separable from the other two species; versicolor and virginica overlap a little, mainly in sepal measurements. Petal length and width are the most informative features.

## 2. Train / test split

In [ ]:
X = iris.data
y = iris.target

# stratify=y keeps the class proportions the same in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

## 3. Train the models

- **Logistic Regression** is a simple linear baseline. It works best with scaled features, so it is wrapped in a pipeline with `StandardScaler`.
- **Random Forest** is an ensemble of decision trees and needs no scaling.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
    ),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")

## 4. Evaluate on the held-out test set

In [ ]:
results = {}
for name, model in models.items():
    preds = model.predict(X_test)
    results[name] = accuracy_score(y_test, preds)

pd.Series(results, name="Test accuracy").round(3).to_frame()

In [ ]:
# Detailed per-class metrics for the Random Forest
best = models["Random Forest"]
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=iris.target_names, cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 5. Cross-validation

A single 80/20 split on only 150 samples can be noisy. 5-fold cross-validation gives a more reliable estimate.

In [ ]:
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    cv_rows.append({"model": name, "mean accuracy": scores.mean(), "std": scores.std()})

pd.DataFrame(cv_rows).round(3)

### Which features matter most?

In [ ]:
importances = pd.Series(best.feature_importances_, index=iris.feature_names).sort_values()

importances.plot.barh(figsize=(6, 3), color="steelblue")
plt.title("Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 6. Predict a new flower

In [ ]:
# sepal length, sepal width, petal length, petal width (cm)
new_flower = pd.DataFrame([[5.9, 3.0, 5.1, 1.8]], columns=iris.feature_names)

pred = best.predict(new_flower)[0]
proba = best.predict_proba(new_flower)[0]

print(f"Predicted species: {iris.target_names[pred]}")
for species, p in zip(iris.target_names, proba):
    print(f"  {species:<11} {p:.1%}")

## Next steps

- Swap in another dataset, e.g. `load_digits()` (handwritten digits) or the Titanic data from `seaborn.load_dataset("titanic")`. Titanic needs missing-value handling and categorical encoding first.
- Tune hyperparameters with `GridSearchCV`.
- Try other models such as `SVC`, `KNeighborsClassifier`, or `GradientBoostingClassifier`.